In [4]:
from pathlib import Path
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


In [5]:


DATA_DIR = Path.cwd().parent/ "data"

METADATA_FILE = DATA_DIR / "ptbxl_database.csv"
STATEMENTS_FILE = DATA_DIR / "scp_statements.csv"

print("Dataset directory:", DATA_DIR)
print("Metadata exists:", METADATA_FILE.exists())
print("Diagnostic statements exist:", STATEMENTS_FILE.exists())

Dataset directory: /Users/priankamondal/Projects/CardioTrustAI/src/CardioTrustAI.Backend/CardioTrustAI.PythonAPI/data
Metadata exists: True
Diagnostic statements exist: True


In [24]:

df = pd.read_csv(METADATA_FILE)
scp = pd.read_csv(STATEMENTS_FILE, index_col=0)

In [34]:
print( df['scp_codes'].head(20))


0       {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
1                   {'NORM': 80.0, 'SBRAD': 0.0}
2                     {'NORM': 100.0, 'SR': 0.0}
3                     {'NORM': 100.0, 'SR': 0.0}
4                     {'NORM': 100.0, 'SR': 0.0}
5                     {'NORM': 100.0, 'SR': 0.0}
6                     {'NORM': 100.0, 'SR': 0.0}
7         {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
8                     {'NORM': 100.0, 'SR': 0.0}
9                     {'NORM': 100.0, 'SR': 0.0}
10                  {'NORM': 80.0, 'SARRH': 0.0}
11                  {'NORM': 80.0, 'SBRAD': 0.0}
12                    {'NORM': 100.0, 'SR': 0.0}
13                    {'NORM': 100.0, 'SR': 0.0}
14                 {'NORM': 100.0, 'SARRH': 0.0}
15                    {'NORM': 100.0, 'SR': 0.0}
16    {'AFLT': 100.0, 'ABQRS': 0.0, 'AFIB': 0.0}
17                               {'AFLT': 100.0}
18                    {'NORM': 100.0, 'SR': 0.0}
19                 {'AFLT': 100.0, 'ABQRS': 0.0}
Name: scp_codes, dty

In [8]:
display(scp.head(10))

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
LVH,left ventricular hypertrophy,1.0,NaN,NaN,HYP,LVH,Ventricular Hypertrophy,left ventricular hypertrophy,142.0,NaN,C71076,NaN
LAFB,left anterior fascicular block,1.0,NaN,NaN,CD,LAFB/LPFB,Intraventricular and intra-atrial Conduction d...,left anterior fascicular block,101.0,MDC_ECG_BEAT_BLK_ANT_L_HEMI,C62267,D3-33140
ISC_,non-specific ischemic,1.0,NaN,NaN,STTC,ISC_,Basic roots for coding ST-T changes and abnorm...,ischemic ST-T changes,226.0,NaN,NaN,NaN


In [30]:
print(df.columns.tolist())

['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


In [10]:
print(scp["diagnostic"].value_counts(dropna=False))

diagnostic
1.0    44
NaN    27
Name: count, dtype: int64


In [11]:
print(scp[["description", "diagnostic", "diagnostic_class"]].head(5))

                          description  diagnostic diagnostic_class
NDT    non-diagnostic T abnormalities         1.0             STTC
NST_          non-specific ST changes         1.0             STTC
DIG                  digitalis-effect         1.0             STTC
LNGQT                long QT-interval         1.0             STTC
NORM                       normal ECG         1.0             NORM


In [12]:
diagnostic_classes = scp[
    scp["diagnostic"] == 1
]["diagnostic_class"].value_counts(dropna=False)

In [39]:
print(diagnostic_classes)

diagnostic_class
MI      14
STTC    13
CD      11
HYP      5
NORM     1
Name: count, dtype: int64


In [14]:
print(df[["ecg_id", "patient_id", "age", "sex", "scp_codes"]].head(10))


   ecg_id  patient_id   age  sex                                 scp_codes
0       1     15709.0  56.0    1  {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
1       2     13243.0  19.0    0              {'NORM': 80.0, 'SBRAD': 0.0}
2       3     20372.0  37.0    1                {'NORM': 100.0, 'SR': 0.0}
3       4     17014.0  24.0    0                {'NORM': 100.0, 'SR': 0.0}
4       5     17448.0  19.0    1                {'NORM': 100.0, 'SR': 0.0}
5       6     19005.0  18.0    1                {'NORM': 100.0, 'SR': 0.0}
6       7     16193.0  54.0    0                {'NORM': 100.0, 'SR': 0.0}
7       8     11275.0  48.0    0    {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
8       9     18792.0  55.0    0                {'NORM': 100.0, 'SR': 0.0}
9      10      9456.0  22.0    1                {'NORM': 100.0, 'SR': 0.0}


In [35]:
print(df["scp_codes"])

0                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
1                             {'NORM': 80.0, 'SBRAD': 0.0}
2                               {'NORM': 100.0, 'SR': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21794    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21795             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21796                           {'ISCAS': 50.0, 'SR': 0.0}
21797                           {'NORM': 100.0, 'SR': 0.0}
21798                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: str


In [44]:
# Convert scp_codes from text to Python dictionaries
df["scp_codes"] = df["scp_codes"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


# Get only diagnostic statements
diagnostic_scp = scp[scp["diagnostic"] == 1]

# Map each ECG's diagnostic codes to their diagnostic classes
def get_diagnostic_classes(codes):
    classes = set()

    for code in codes:
        if code in diagnostic_scp.index:
            diagnostic_class = diagnostic_scp.loc[code, "diagnostic_class"]

            if isinstance(diagnostic_class, str):
                classes.add(diagnostic_class)

    return sorted(classes)

df["diagnostic_classes"] = df["scp_codes"].apply(get_diagnostic_classes)



In [17]:
print("Total ECG records:", len(df))

print("\nDiagnostic class counts:")
print(
    df["diagnostic_classes"]
    .explode()
    .value_counts()
)

print("\nECGs without diagnostic classes:")
print(
    df["diagnostic_classes"]
    .apply(len)
    .eq(0)
    .sum()
)

Total ECG records: 21799

Diagnostic class counts:
diagnostic_classes
NORM    9514
MI      5469
STTC    5235
CD      4898
HYP     2649
Name: count, dtype: int64

ECGs without diagnostic classes:
411


In [46]:
# Select ECGs without diagnostic classes
unmapped_ecgs = df[
    df["diagnostic_classes"].apply(len) == 0
]

print("ECGs without diagnostic classes:", len(unmapped_ecgs))

# Show their original diagnostic codes
print(
    unmapped_ecgs[
        ["ecg_id", "scp_codes"]
    ].head(10)
)

ECGs without diagnostic classes: 411
     ecg_id                                   scp_codes
16       17  {'AFLT': 100.0, 'ABQRS': 0.0, 'AFIB': 0.0}
17       18                             {'AFLT': 100.0}
19       20               {'AFLT': 100.0, 'ABQRS': 0.0}
22       23                             {'AFLT': 100.0}
33       34                             {'AFLT': 100.0}
137     144                             {'PACE': 100.0}
182     190                             {'PACE': 100.0}
374     382                             {'PACE': 100.0}
377     385                             {'PACE': 100.0}
441     449                             {'AFLT': 100.0}


In [15]:
# Inspect the codes in the first 10 unmapped ECGs
for codes in unmapped_ecgs["scp_codes"].head(10):
    for code in codes:
        if code not in diagnostic_scp.index:
            if code in scp.index:
                print(
                    code,
                    "| diagnostic:",
                    scp.loc[code, "diagnostic"],
                    "| class:",
                    scp.loc[code, "diagnostic_class"]
                )
            else:
                print(code, "| not found in scp table")

AFLT | diagnostic: nan | class: nan
ABQRS | diagnostic: nan | class: nan
AFIB | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
ABQRS | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan


In [16]:
patient_counts = (
    df.groupby("strat_fold")["patient_id"]
    .nunique()
    .sort_index()
)

print("Unique patients in each fold:")
print(patient_counts)

Unique patients in each fold:
strat_fold
1     1878
2     1849
3     1887
4     1883
5     1890
6     1884
7     1871
8     1881
9     1942
10    1904
Name: patient_id, dtype: int64


In [17]:
train_df = df[df["strat_fold"].between(1, 8)].copy()
val_df = df[df["strat_fold"] == 9].copy()
test_df = df[df["strat_fold"] == 10].copy()

print("Training ECGs:", len(train_df))
print("Validation ECGs:", len(val_df))
print("Testing ECGs:", len(test_df))

print("\nUnique patients:")
print("Training:", train_df["patient_id"].nunique())
print("Validation:", val_df["patient_id"].nunique())
print("Testing:", test_df["patient_id"].nunique())

Training ECGs: 17418
Validation ECGs: 2183
Testing ECGs: 2198

Unique patients:
Training: 15023
Validation: 1942
Testing: 1904


In [22]:
train_patients = set(train_df["patient_id"].dropna())
val_patients = set(val_df["patient_id"].dropna())
test_patients = set(test_df["patient_id"].dropna())

print("Training and validation overlap:",
      len(train_patients & val_patients))

print("Training and testing overlap:",
      len(train_patients & test_patients))

print("Validation and testing overlap:",
      len(val_patients & test_patients))

Training and validation overlap: 0
Training and testing overlap: 0
Validation and testing overlap: 0


In [23]:
def count_diagnostic_classes(data):
    return (
        data["diagnostic_classes"]
        .explode()
        .value_counts()
        .reindex(["NORM", "MI", "STTC", "CD", "HYP"], fill_value=0)
    )

print("TRAINING CLASS COUNTS")
print(count_diagnostic_classes(train_df))

print("\nVALIDATION CLASS COUNTS")
print(count_diagnostic_classes(val_df))

print("\nTESTING CLASS COUNTS")
print(count_diagnostic_classes(test_df))

TRAINING CLASS COUNTS
diagnostic_classes
NORM    7596
MI      4379
STTC    4186
CD      3907
HYP     2119
Name: count, dtype: int64

VALIDATION CLASS COUNTS
diagnostic_classes
NORM    955
MI      540
STTC    528
CD      495
HYP     268
Name: count, dtype: int64

TESTING CLASS COUNTS
diagnostic_classes
NORM    963
MI      550
STTC    521
CD      496
HYP     262
Name: count, dtype: int64
